# Notebook 016: Struct Creation, Nested Field Access and Updating Sub-Fields

**Module:** Module 02: Complex Data Types & Nested Structures Handling  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 14: Master Unity Catalog & Video 23: Interview Questions  

---

## 1. Business Scenario & Real-World Context
A customer identity platform groups personal attributes, contact points, and address details into nested structs for clean schema organization. Downstream systems need to query specific sub-fields and update an address without reconstructing the entire object.

## 2. Technical Requirement & Problem Statement
Create nested `StructType` columns using `struct()`. Access inner fields via dot notation and `col.getField()`. Demonstrate updating a single nested field inside a struct in PySpark 3.1+ without manually re-specifying all sibling fields.

## 3. Spark & Databricks Architecture Notes
StructType is a compound type that stores multiple typed named fields together in memory. In earlier Spark versions, updating one nested field required re-creating the entire struct. Spark 3.1+ introduced `withField()` and `dropFields()` on Struct columns to manipulate nested fields cleanly.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_016_Struct_Creation,_Nested_Field_Access_and_Updating_Sub-Fields")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Creating Nested Struct Records
Package flat customer columns into organized nested structs.

In [ ]:
customers_flat = [
    ("C-01", "Alice", "Smith", "123 Elm St", "Austin", "TX", "78701", 85000),
    ("C-02", "Bob", "Jones", "456 Oak Ave", "Seattle", "WA", "98101", 92000)
]

cols = ["id", "first_name", "last_name", "street", "city", "state", "zip", "salary"]
df_flat = spark.createDataFrame(customers_flat, cols)

# Bundle into nested structs
df_nested = df_flat.select(
    F.col("id"),
    F.struct(F.col("first_name"), F.col("last_name")).alias("name"),
    F.struct(F.col("street"), F.col("city"), F.col("state"), F.col("zip")).alias("address"),
    F.col("salary")
)

df_nested.printSchema()
df_nested.show(truncate=False)

## 6. Querying Nested Fields via Dot Notation & getField
Select specific sub-fields directly.

In [ ]:
# Accessing nested fields
df_selected = df_nested.select(
    "id",
    "name.last_name",
    F.col("address.city").alias("home_city"),
    F.col("address").getField("state").alias("home_state")
)

df_selected.show()

## 7. Updating Single Nested Fields using withField (Spark 3.1+)
Modify address.city without rewriting the entire struct definition.

In [ ]:
# Update city to uppercase and add a new nested field country
df_updated_struct = df_nested.withColumn(
    "address",
    F.col("address")
     .withField("city", F.upper(F.col("address.city")))
     .withField("country", F.lit("USA"))
)

df_updated_struct.printSchema()
df_updated_struct.select("id", "address").show(truncate=False)

## 9. Key Architectural Takeaways
- **Use `struct(*cols)` to group related columns into a compound object without network shuffle.**
- **`col.withField('subfield', expr)` (Spark 3.1+) updates or adds a nested field inside an existing struct.**
- **`col.dropFields('subfield')` removes specific nested attributes from a struct.**

## 10. Senior Data Engineering Interview Questions
1. How does `withField()` simplify nested struct manipulation compared to older Spark versions?
2. What is the difference between dot notation `col('a.b')` and `col('a').getField('b')`?
3. Can Parquet push down column projection into individual fields of a nested StructType?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
